In [ ]:
import pandas as pd
import requests

owner = "INT-D-491-W26"
repo = "project"
branch = "main"
base_folder = "cleaned_data_files"

all_csv_urls = []

def get_csv_urls(folder_path):
    url = f"https://api.github.com/repos/{owner}/{repo}/contents/{folder_path}?ref={branch}"
    response = requests.get(url)
    items = response.json()

    for item in items:
        if item["type"] == "file" and item["name"].endswith(".csv"):
            all_csv_urls.append(item["download_url"])
        elif item["type"] == "dir":
            get_csv_urls(item["path"])

get_csv_urls(base_folder)

print("Total CSV files found:", len(all_csv_urls))

dfs = []
for file_url in all_csv_urls:
    print("Loading:", file_url)
    df = pd.read_csv(file_url)

    if "cleaned_y_a" in file_url:
      df["age_group"] = "young_adults"
    elif "cleaned_2550" in file_url:
      df["age_group"] = "adults_25_50"
    elif "cleaned_50up" in file_url:
      df["age_group"] = "adults_50_up"

    if "rural" in file_url:
      df["area_type"] = "rural"
    elif "urban" in file_url:
      df["area_type"] = "urban"

    dfs.append(df)

combined_df = pd.concat(dfs, ignore_index=True)
print("Final combined shape:", combined_df.shape)

combined_df.to_csv("combined_cleaned_data.csv", index=False)
print("Combined file saved as combined_cleaned_data.csv")

Total CSV files found: 54
Loading: https://raw.githubusercontent.com/INT-D-491-W26/project/main/cleaned_data_files/cleaned_2550_f_r/adults_2550_female_rural_0000-0399_clean.csv
Loading: https://raw.githubusercontent.com/INT-D-491-W26/project/main/cleaned_data_files/cleaned_2550_f_r/adults_2550_female_rural_0400-0799_clean.csv
Loading: https://raw.githubusercontent.com/INT-D-491-W26/project/main/cleaned_data_files/cleaned_2550_f_r/adults_2550_female_rural_0800-1199_clean.csv
Loading: https://raw.githubusercontent.com/INT-D-491-W26/project/main/cleaned_data_files/cleaned_2550_f_r/adults_2550_female_rural_1600-1999_clean.csv
Loading: https://raw.githubusercontent.com/INT-D-491-W26/project/main/cleaned_data_files/cleaned_2550_f_u/adults_2550_female_urban_0000-0399_clean.csv
Loading: https://raw.githubusercontent.com/INT-D-491-W26/project/main/cleaned_data_files/cleaned_2550_f_u/adults_2550_female_urban_0400-0799_clean.csv
Loading: https://raw.githubusercontent.com/INT-D-491-W26/project/mai

In [ ]:
combined_df['gender'].value_counts()
combined_df['gender'].value_counts(normalize=True) * 100

,proportion
gender,
F,52.093631
M,47.906369


Our dataset contains 52.09% females and 47.91% males, indicating that both gender groups are nearly equally represented.

Relative to the overall dataset size, this difference of about 4% is very small. Hence, there's no evidence of gender-based bias, and no corrective actions are required.

However, to maintain fairness during model development, the following practices can be adopted:

1. Stratified sampling when splitting training and testing datasets to preserve gender proportions.

2. Monitoring model performance by gender to ensure predictive outcomes remain equitable.

In [ ]:
combined_df.groupby('gender')['is_fraud'].mean()

,is_fraud
gender,
F,0.005298
M,0.005658


We see extremely small differences in average fraud rates of 0.53% for females, and 0.57% for males. Such a minor difference is unlikely to show systematic bias and instead reflects natural variation in transaction behavior.

There is no disparity in fraud labellings between genders, and no immediate corrective steps are required.

In [ ]:
combined_df["age_group"].value_counts()
combined_df["age_group"].value_counts(normalize=True) * 100

,proportion
age_group,
adults_25_50,55.763963
adults_50_up,32.582723
young_adults,11.653314


We see a clear imbalance between our data, where individuals between 25-50 years old are dominant, and young adults are underrepresented.

This likely reflects real-world financial demographics, and doesn't necessarily indicate a dataset bias.

However, since the young adults group is substantially smaller, models trained on this data may have less exposure to behavior patterns from this demographic.



In [ ]:
combined_df.groupby("age_group")["is_fraud"].mean()

,is_fraud
age_group,
adults_25_50,0.003895
adults_50_up,0.008286
young_adults,0.005138


We see that fraud incidence increases with age. This pattern aligns with documented financial fraud trends, where older individuals are often more vulnerable to fraud attempts.

Factors such as reduced familiarity with emerging digital scams or targeted social engineering attacks, may play a big role in this.



In [ ]:
combined_df.groupby(["gender","age_group"])["is_fraud"].mean()

gender  age_group   
F       adults_25_50    0.003582
        adults_50_up    0.008517
        young_adults    0.005069
M       adults_25_50    0.004256
        adults_50_up    0.008053
        young_adults    0.005209
Name: is_fraud, dtype: float64

Differences between genders within the same age group are relatively small.

However, fraud rates increased for the 50+ age group for both genders, while younger groups show relatively lower and similar rates.

1. Stratified Sampling ensuring train-test splits preserve proportional representation of subgroups.

2. Performance Evaluation by grouping, evaluating model accuracy and error rates separately for each age & gender subgroup to detect potential disparities.

In [ ]:
combined_df["area_type"].value_counts(normalize=True) * 100

,proportion
area_type,
urban,94.911885
rural,5.088115


The dataset contains a strong geographic representation imbalance, with approx. 95% of transactions from urban areas and only about 5% from rural areas.

This likely reflects real-world economic activity patterns, as urban areas typically have higher population density and greater usage of credit cards and digital payment systems.

While this does not necessarily indicate bias in fraud labeling, models trained on this dataset may primarily capture patterns from urban transactions, which could reduce predictive accuracy for rural populations.



In [ ]:
combined_df.groupby("area_type")["is_fraud"].mean()

,is_fraud
area_type,
rural,0.004802
urban,0.005506


In [ ]:
combined_df.groupby(["area_type","age_group"])["is_fraud"].mean()

area_type  age_group   
rural      adults_25_50    0.004921
           adults_50_up    0.004875
           young_adults    0.004150
urban      adults_25_50    0.003853
           adults_50_up    0.008536
           young_adults    0.005197
Name: is_fraud, dtype: float64

Generally, similar fraud rates between rural and urban areas, among different age groups suggest no strong geographic biases.

However, **urban adults aged 50+** show a higher fraud rate (0.85%) compared to other groups (0.4-0.5%), indicating this demographic is more frequently targeted or vulnerable to fraud.

Potential Mitigation Strategies:
1. Stratified Sampling ensuring rural and urban observations are proportionally represented in both training and testing datasets.

2. Subgroup model evaluation separately for age and geographic subgroups to ensure predictions remain consistent across all subgroups.

In [ ]:
combined_df.groupby("category")["is_fraud"].mean().sort_values(ascending=False)

,is_fraud
category,
shopping_net,0.017384
misc_net,0.015194
grocery_pos,0.013518
shopping_pos,0.006601
gas_transport,0.005150
travel,0.002868
misc_pos,0.002566
entertainment,0.002230
personal_care,0.002044


Overall, online shopping transactions (shopping_net) exhibit the highest fraud risk, followed by other internet-based transaction types.

Again, these results reflect realistic fraud patterns observed in financial systems, where fraud is significantly higher in online transactions, and miscellaneous online payments.

This is mainly due to lack of physical card usage which involves remote payment processing. Differences arise from transaction characteristics reflecting behavioral risk patterns.

Potential Risks for ML Models: If a model learns that certain categories strongly correlate with fraud, it may overly rely on transaction categories.

Mitigation Strategies:
1. Feature Regularization to prevent models relying excessively on any single feature.
2. Balanced Model Evalutions to separately detect accuracy/recall for transactions category.
3. Threshold Calibration to reduce excessive false positives in high-risk categories.
4. Cross-feature analysis with demographic variables ensuring that category effects do not indirectly introduce demographic bias.